### Import libraries and also define variables

In [34]:
#Mitchell Moffitt
#Preprocessign done - 10/11/2026


import fennec_ml as fn                     # Folder cleaner, code posted on github: also use pip install fennec_ml to use
import torch                               # PyTorch: tensors, GPU support, and automatic gradients for training
import torch.nn as nn                      # building blocks for models: layers (Linear, Conv1d, LSTM) and losses (MSELoss)
import torch.nn.functional as F            # the same operations as plain functions: F.relu, F.mse_loss, F.dropout
import os                                  # file paths and folders: os.path.join, os.listdir, os.makedirs
import json                                # save and load settings or results as .json (config, metrics, label maps)
from torch.utils.data import TensorDataset, DataLoader
                                           # TensorDataset pairs each window with its target;
                                           # DataLoader hands them to the model in shuffled batches
import glob                                # find files by pattern, e.g. glob.glob("Raw_data/*.xlsx")
import re                                  # regular expressions, used to check filename formats like "0.15A"
import matplotlib.pyplot as plt            # plotting: loss curves, predicted vs true CG, flight signals

#Check Pytorch Version
torch.__version__

'2.14.0+cpu'

### Define Variables

In [35]:
#Variables (Hyperparameters)

seed = 42 #For reproducibility

#Sets the amount of neurons per layer
HIDDEN_LAYER1 = 10
HIDDEN_LAYER2 = 10
HIDDEN_LAYER3 = 10  #I dont use htis but more can be added

OUTPUT_FEATURES = 1 #Number of output features (CG prediction in 1D)

EPOCHS = 10  # Number of training epochs
LR = 0.01
WEIGHT_DECAY = 1e-2 #HOW SHOULD I SET THIS?
DROPOUT_RATE1 = 0.1
DROPOUT_RATE2 = 0.3
SEGMENT_LENGTH = 50  # Length of each segment for the flight data


### Processing data

Open folder config and load the necessary configuration files for data processing.

In [36]:
# --- FOLDER CONFIG ---
data_dir = os.path.join(os.getcwd(), "testing_data")
os.makedirs(data_dir, exist_ok=True)

#MAKE SURE THESE ARE CORRECT
excel_dir = data_dir = r"C:\Users\Mitch\OneDrive - LeTourneau University\Andrew Davis's files - Project FENNEC\FENNEC 2026-2027\Data Science Team\Mitchell Moffitt\Python Code\Jupyter Notebooks\GOAL 3- Data preprocessing\Raw_data"
csv_dir = r"C:\Users\Mitch\OneDrive - LeTourneau University\Andrew Davis's files - Project FENNEC\FENNEC 2026-2027\Data Science Team\Mitchell Moffitt\Python Code\Jupyter Notebooks\GOAL 3- Data preprocessing\Processed_data"

# --- MAKE VARS OF INTEREST ---
base_dir = os.getcwd()
json_path = os.path.join(base_dir, "1D_Variable_vars_of_interest.json")

# Data to write - add or remove as needed
INPUT_FEATURES = 6 # HOW MANY ARE IN VARS OF INTEREST?
vars_of_interest = {
    "IMU_0": ['GyrX', 'GyrY', 'GyrZ', 'AccX', 'AccY', 'AccZ'],
    #"RCOU": ['C1','C2','C3','C4'],
    #"ATT": ["Roll","Pitch","Yaw"],
    #"ATT": ["DesRoll","Roll","DesPitch","Pitch","DesYaw","Yaw","ErrRP","ErrYaw"],
    #"RATE": ["RDes","R","ROut","PDes","P","POut","YDes","Y","YOut"], # Not sure but could have been 88% acuracy
    #"VIBE_0": ["VibeX", "VibeY", "VibeZ"],
    #"VIBE_1": ["VibeX", "VibeY", "VibeZ"]
}

# Write to JSON file
with open(json_path, "w") as f:
    json.dump(vars_of_interest, f, indent=4)

print(f"Created {json_path}")

Created c:\Users\Mitch\OneDrive - LeTourneau University\Andrew Davis's files - Project FENNEC\FENNEC 2026-2027\Data Science Team\Mitchell Moffitt\Python Code\Jupyter Notebooks\GOAL 4 - Put code together\1D_Variable_vars_of_interest.json


* Write to CSV file
* also cleans up data so that the model cannot get confused by all of the other data that a flight controller collects

In [37]:
# --- EXCEL TO CSV ---
#Remember to clear out the old data before writing more to the processed data
print("Using folder_cleaner() to convert from raw excel files to useful .csv's.\nfolder_cleaner() output:")

print("Running folder_cleaner...")
#Takes from the raw excel files in excel_dir and writes processed CSVs to csv_dir
fn.folder_cleaner(excel_dir, csv_dir, overwrite= True, varspath="1D_Variable_vars_of_interest.json", downsample=False)
print("folder_cleaner finished")

Using folder_cleaner() to convert from raw excel files to useful .csv's.
folder_cleaner() output:
Running folder_cleaner...
100B_0.75F_H.xlsx processed and saved to C:\Users\Mitch\OneDrive - LeTourneau University\Andrew Davis's files - Project FENNEC\FENNEC 2026-2027\Data Science Team\Mitchell Moffitt\Python Code\Jupyter Notebooks\GOAL 3- Data preprocessing\Processed_data as 100B_0.75F_H.csv
101B_0.60A_M.xlsx processed and saved to C:\Users\Mitch\OneDrive - LeTourneau University\Andrew Davis's files - Project FENNEC\FENNEC 2026-2027\Data Science Team\Mitchell Moffitt\Python Code\Jupyter Notebooks\GOAL 3- Data preprocessing\Processed_data as 101B_0.60A_M.csv
102B_0.75A_M.xlsx processed and saved to C:\Users\Mitch\OneDrive - LeTourneau University\Andrew Davis's files - Project FENNEC\FENNEC 2026-2027\Data Science Team\Mitchell Moffitt\Python Code\Jupyter Notebooks\GOAL 3- Data preprocessing\Processed_data as 102B_0.75A_M.csv
103B_0.15A_M.xlsx processed and saved to C:\Users\Mitch\OneDriv

This Section uses the Variable_CG_Preprocessing file I created

It does these things:

* Splits data into train/validation/test set for appropriate model training
* Imports normalizer so that the differing units do nto skew the data such as the gyro adn the accelrometer have different magnitubes of numbers and it breaks it down into the mean the std so that the data will not be skewed
* Then it segments the data, since there are so many timesteps per flight this makes it so that the model trains on evenly split portions or windows of data (which is needed), it will divide the data up into a certain amount of timesteps and send that chunk of data into the model
* The data loader feeds your windows to the model in small batches during training. It does the bookkeeping that you would otherwise write by hand.

### Augmentation/ noise

Should this be added?

In [38]:

from Variable_CG_Preprocessing import Flight_Objects_Var, FlightDataNormalizer_Var, FlightDataSegmenter_Var
from collections import Counter
from Variable_CG_Preprocessing import split_flights_by_fraction, make_dataloader_Var

flights1 = Flight_Objects_Var(csv_dir)          # load once

counts = Counter(f.cg_code for f in flights1.flights)
minimum = min(counts.values())

#This does an 80-0-20 split of the flights into train, val, and test sets
#This might be too much data? if with a large enough data set
train_frac = 0.8
val_frac = 0.0

train_flights, val_flights, test_flights = split_flights_by_fraction(
    flights1, train_frac=train_frac, val_frac=val_frac
)

print("Number of train flights:", len(train_flights))
print("Number of val flights:", len(val_flights))
print("Number of test flights:", len(test_flights))

# Data normilzer
normalizer = FlightDataNormalizer_Var()

# Fit norm on train data, then apply to val/test
train_flights = normalizer.fit_transform(train_flights)

# Transfrom to Val/Test
val_flights = normalizer.transform(val_flights)
test_flights = normalizer.transform(test_flights)

# Justin had it at 50 and a medium number like this makes sense not too much at once
segment_length = SEGMENT_LENGTH       # keep whatever you set earlier
stride = segment_length     # no overlap; use segment_length // 2 for 50% overlap

def n_windows(flight):
    """How many windows fit in this flight at the chosen length and stride."""
    n = flight.data.shape[0]
    return (n - segment_length) // stride + 1 if n >= segment_length else 0

# Smallest flight (in windows) in each split, used to balance flights within the split
train_min_segments_per_flight = min(n_windows(f) for f in train_flights if n_windows(f) > 0)
test_min_segments_per_flight = min(n_windows(f) for f in test_flights if n_windows(f) > 0)

print("Smallest flight in training set (windows):", train_min_segments_per_flight)
print("Smallest flight in testing set (windows):", test_min_segments_per_flight)

# Separate segmenters so each split is internally balanced
train_segmenter = FlightDataSegmenter_Var(
    segment_length=segment_length,
    stride=stride,
    max_segments_per_flight=train_min_segments_per_flight,
    seed=42,
)
test_segmenter = FlightDataSegmenter_Var(
    segment_length=segment_length,
    stride=stride,
    max_segments_per_flight=test_min_segments_per_flight,
    seed=42,
)

# Returns: windows, scaled CG targets, and which flight each window came from
train_segments, train_targets, train_flight_idx = train_segmenter.segment_flights(train_flights)
test_segments, test_targets, test_flight_idx = test_segmenter.segment_flights(test_flights)

# Check shapes
print("Train segments:", train_segments.shape)    # (N, L, F)
print("Train targets: ", train_targets.shape)     # (N,)
print("Test segments: ", test_segments.shape)
print("Test targets:  ", test_targets.shape)

# Data loaders
batch_size = 32

train_loader = make_dataloader_Var(train_segments, train_targets, batch_size=batch_size, shuffle=True)
test_loader = make_dataloader_Var(test_segments, test_targets, batch_size=batch_size, shuffle=False)

print(f"Number of batches in train loader: {len(train_loader)}")
print(f"Number of batches in test loader: {len(test_loader)}")

#if ur seeing this then ur cool - mitchell

Number of train flights: 110
Number of val flights: 0
Number of test flights: 22
Scaler fitted: mean shape (6,) var shape (6,)
Scaler fitted on training data:
Feature 0: mean = -0.0013, std = 0.3422
Feature 1: mean = -0.0009, std = 0.3548
Feature 2: mean = -0.0068, std = 0.4248
Feature 3: mean = -1.2597, std = 1.8520
Feature 4: mean = -1.2271, std = 1.9750
Feature 5: mean = -9.5041, std = 1.5134
Smallest flight in training set (windows): 44
Smallest flight in testing set (windows): 44
Train segments: (4840, 50, 6)
Train targets:  (4840,)
Test segments:  (968, 50, 6)
Test targets:   (968,)
Number of batches in train loader: 152
Number of batches in test loader: 31


This code below is basically so that the whole entire data set does not need to be used at once and the model can be trained on smaller data sets like 44 training flights and 11 test flights rather than an 80-0-20 split on over 100 data points

In [39]:
#EXTRA CODE DO NOT RUN

# from Variable_CG_Preprocessing import split_flights_by_class

# train_per_class = 3 #asks for 3 flights per CG position (so 11 postions * 3 = 33 flights for training)
# val_per_class = 0 #asks for 0 flights per CG position for validation

# train_flights, val_flights, test_flights = split_flights_by_class(flights1,train_per_class=train_per_class, val_per_class=val_per_class)
# print(f"Number of training flights: {len(train_flights)}")
# print(f"Number of val flights: {len(val_flights)}")
# print(f"Number of testing flights: {len(test_flights)}")

This tells us the duration fo each flight with full segments

In [40]:
n_show = 5   # how many flights to print from each split

print(f"Train: showing {n_show} of {len(train_flights)} flights")
for i, f in enumerate(train_flights[:n_show]):
    print(f"  Flight {i} ({f.filename}): {f.data.shape[0] // segment_length} full segments")

print(f"\nTest: showing {n_show} of {len(test_flights)} flights")
for i, f in enumerate(test_flights[:n_show]):
    print(f"  Flight {i} ({f.filename}): {f.data.shape[0] // segment_length} full segments")

Train: showing 5 of 110 flights
  Flight 0 (185B_0.75F_H.csv): 44 full segments
  Flight 1 (171B_0.75F_L.csv): 44 full segments
  Flight 2 (135B_0.75F_M.csv): 44 full segments
  Flight 3 (195B_0.75F_H.csv): 44 full segments
  Flight 4 (200B_0.75F_L.csv): 44 full segments

Test: showing 5 of 22 flights
  Flight 0 (133B_0.75F_M.csv): 44 full segments
  Flight 1 (201B_0.75F_L.csv): 44 full segments
  Flight 2 (224B_0.60A_M.csv): 44 full segments
  Flight 3 (101B_0.60A_M.csv): 44 full segments
  Flight 4 (178B_0.75A_H.csv): 44 full segments


Checks Normalization: Mean and Standard Deviation

In [41]:
import numpy as np

# X_train_segments is your list/array of training segments (num_segments, seq_len, num_features)
X_train = np.concatenate(train_segments, axis=0)  # flatten over segments

# Compute per-feature mean and std
feature_means = X_train.mean(axis=0)
feature_stds  = X_train.std(axis=0)

print("Per-feature mean after normalization:", feature_means)
print("Per-feature std after normalization:", feature_stds, "\n")

# Overall mean/std
print(f"Mean should be close to 0 and std should be close to 1")
print("Overall mean:", X_train.mean())
print("Overall std:", X_train.std())

Per-feature mean after normalization: [-8.2365368e-05 -4.5623274e-05 -3.5838937e-04 -8.9397877e-03
 -9.8508867e-03  4.1330764e-03]
Per-feature std after normalization: [1.010983  1.0109911 1.0110703 1.009138  1.0086361 1.0105792] 

Mean should be close to 0 and std should be close to 1
Overall mean: -0.0025239962
Overall std: 1.0102463


### Check data segmentation
* make sure each offset has equal representation

In [42]:
from collections import Counter
from Variable_CG_Preprocessing import unscale_target

def cg_distribution(targets_scaled):
    """Count windows per CG offset (inches), sorted aft (-) to forward (+)."""
    inches = np.round(unscale_target(targets_scaled), 2)
    return dict(sorted(Counter(inches.tolist()).items()))

print("Train CG distribution (windows per offset, in):")
for cg, n in cg_distribution(train_targets).items():
    print(f"  {cg:+.2f}: {n}")

print("\nTest CG distribution (windows per offset, in):")
for cg, n in cg_distribution(test_targets).items():
    print(f"  {cg:+.2f}: {n}")

Train CG distribution (windows per offset, in):
  -0.75: 440
  -0.60: 440
  -0.45: 440
  -0.30: 440
  -0.15: 440
  +0.00: 440
  +0.15: 440
  +0.30: 440
  +0.45: 440
  +0.60: 440
  +0.75: 440

Test CG distribution (windows per offset, in):
  -0.75: 88
  -0.60: 88
  -0.45: 88
  -0.30: 88
  -0.15: 88
  +0.00: 88
  +0.15: 88
  +0.30: 88
  +0.45: 88
  +0.60: 88
  +0.75: 88


### Build model

- nn.Sequential: This model basically goes in order of what you put into it from top to bottom
- nn.Conv1d: It slides a small set of learned weights along a sequence, computing a weighted sum over a few neighboring time steps at each position, so it can detect patterns anywhere in the data.
- nn.AdaptiveAvgPool1d: Tt averages a sequence down to a fixed output size that you choose (like 1 value per channel), no matter how long the input is.
- nn.dropout: Each value has a 10% (or whatever you set it to) chance of being zeroed, and a different random set is chosen each time.

    Why it helps: without dropout, a network can come to rely on a few specific neurons that happen to fit your training flights. When any neuron might vanish on the next pass, the network has to spread what it learns across many neurons, so no single one carries too much. That makes it less likely to memorize the training data (overfitting) and more likely to generalize to new flights. You can think of it as training many slightly different smaller networks and averaging them. TYPICAL VALUES FOR SMALLER MODEL ARE BETWEEN 0.1 AND 0.3.


In [43]:
#nn.BatchNorm1d(32) might be worth adding after each conv it must match the conv layer size
#It normalizes the outpts from each layer
class OneDVariableCGPredictionModel(nn.Module):
    def __init__(self, INPUT_FEATURES, HIDDEN_LAYER1, HIDDEN_LAYER2, OUTPUT_FEATURES):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(INPUT_FEATURES, HIDDEN_LAYER1, kernel_size=7, padding=3), 
            nn.ReLU(),
            nn.Dropout1d(DROPOUT_RATE1),
            nn.Conv1d(HIDDEN_LAYER1, HIDDEN_LAYER2, kernel_size=7, padding=3), 
            nn.ReLU(),
            nn.Dropout1d(DROPOUT_RATE1),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)   # collapses time -> 1 value per channel, this allows nn.Linear to work
        self.drop = nn.Dropout(DROPOUT_RATE2) 
        self.head = nn.Linear(HIDDEN_LAYER2, OUTPUT_FEATURES) #ends at one output which is the CG prediction

    def forward(self, x):                      # x: (batch, 6, time)
        x = self.conv(x)
        x = self.pool(x).squeeze(-1)           # (batch, 64)
        x = self.drop(x)
        return self.head(x)                    # (batch, 1) -> predicted CG

The random seed only applies to the random parameters

In [45]:
#Create random seed for reproducibility
torch.manual_seed(seed) #sets intial conditions biscally, create repeatibility

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Create instance of the model (this is a subclass of nn.module)
LRModel = OneDVariableCGPredictionModel(
    INPUT_FEATURES, 
    HIDDEN_LAYER1, 
    HIDDEN_LAYER2, 
    OUTPUT_FEATURES).to(device)


#CHeck out parameters
#list(LRModel.parameters())

#Create loss
loss_fn = nn.MSELoss()

#Create optimizer
#Adam defaults weight decay to 0, so AdamW is used to explicitly set it
optimizer = torch.optim.AdamW(LRModel.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


#This is useful for training plauteaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1, #How much it cuts the lr down by
    patience=10 #Number of epochs with no improvement after which the learning rate will be reduced
)
#The scheduler will reduce the learning rate when the validation loss plateaus
#A validation set would be better to use for this since as of now it uses the test data set

In [46]:
LRModel.eval()  # turns off dropout, uses running stats in batch norm

all_preds, all_targets = [], []

with torch.inference_mode():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.permute(0, 2, 1).to(device)
        preds = LRModel(X_batch.to(device))
        all_preds.append(preds.cpu().numpy())
        all_targets.append(y_batch.numpy())

preds_scaled = np.concatenate(all_preds).ravel()      # in [-1, 1]
targets_scaled = np.concatenate(all_targets).ravel()

# Convert back to inches before computing metrics
preds_in = unscale_target(preds_scaled)
targets_in = unscale_target(targets_scaled)

mae = np.mean(np.abs(preds_in - targets_in))
rmse = np.sqrt(np.mean((preds_in - targets_in) ** 2))
print(f"MAE: {mae:.3f} in | RMSE: {rmse:.3f} in")

MAE: 0.450 in | RMSE: 0.535 in


### Training loop

In [47]:
from unittest import loader

from sympy import evaluate


save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

best_test_full_acc = -1.0
best_epoch = -1
best_path = os.path.join(save_dir, "Best_1DVar_Model.pth")


### Training
    
for epoch in range(EPOCHS):
    LRModel.train()                                   # turns dropout / batch-norm training behavior back on
    total_loss, total_samples = 0.0, 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)                # (B, 1), scaled to [-1, 1]

        optimizer.zero_grad()                       # clear gradients from the previous batch
        pred = LRModel(X_batch).view_as(y_batch)      # (B, 1)
        loss = loss_fn(pred, y_batch)

        loss.backward()                             # compute gradients
        torch.nn.utils.clip_grad_norm_(LRModel.parameters(), clip)   # stops rare huge updates
        
        optimizer.step()                            # update the weights

        bs = X_batch.size(0)
        total_loss += loss.item() * bs
        total_samples += bs

    avg_train_loss = total_loss / total_samples
    train_acc_full = total_correct_full / total_samples

    avg_test_loss, test_acc_full = evaluate(
            LRModel, test_loader, loss_fn, device
    )
    print(
            f"Epoch [{epoch+1}/{EPOCHS}] | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Train Full: {train_acc_full:.4f} | "
            f"Test Loss: {avg_test_loss:.4f} | "
            f"Test Full: {test_acc_full:.4f}"
        )



TypeError: 'module' object is not iterable